# Parametrized gain functions of the sKF-L filters

The two closed-form filters for a Gaussian prior with a Laplacian likelihood are both a scalar
gain times the regressor:

$$\boldsymbol{w}_t = \boldsymbol{w}_{t-1} + g_1(e_t, b_\eta, \tilde v_t, \|\boldsymbol{x}_t\|)\,\boldsymbol{x}_t,
\qquad v_t = \tilde v_t + g_2(e_t, b_\eta, \tilde v_t, \|\boldsymbol{x}_t\|)\,\|\boldsymbol{x}_t\|^2 ,
\qquad \tilde v_t = v_{t-1} + \varepsilon .$$

This notebook fixes $b_\eta$, $\tilde v_t$ and $\|\boldsymbol{x}_t\|$ at the operating point of the
`##### Laplacian Likelihood` cells of `bayes-adaptive-filters.ipynb`, and plots $g_1$ and $g_2$
against $e_t$ for

* **sKF-L minorized**, draft eqs. (50)-(51) — `sKF_L_algorithm`
* **sKF-L exact**, draft eqs. (63)-(64) collapsed to (69)-(70) — `sKF_L_exact_algorithm`
* the **numerical inversion of eq. (18)**, the ground truth

so that a disagreement shows up at the value of $e_t$ where it happens, instead of being smeared
across a 200-step trajectory.

In [ ]:
import numpy as np
from matplotlib import pyplot as plt
from numba import njit
from scipy.special import log_ndtr, logsumexp

from filters import (
    # parameter dtypes
    sKF_L_params, gaussian_params, laplacian_params,
    # densities and the convolution machinery of the integral filters
    gaussian_pdf, laplacian_pdf, integral_convolve_from_base_pdf, _get_integration_range,
    # signal / environment helpers
    shift, std_behavior, AR_settling_time,
    # the two closed forms under test
    sKF_L_algorithm, sKF_L_exact_algorithm,
)

%config InlineBackend.figure_format = 'svg'
COLORS = plt.rcParams['axes.prop_cycle'].by_key()['color']

## 1. Operating point

Everything below is taken from the `##### Laplacian Likelihood` cells of
`bayes-adaptive-filters.ipynb`: `M = 3`, `var_x = 1`, `var_v = 1e-3`,
`b_eta = 5*sqrt(var_v)`, `epsilon = 0.01`, `AR = [1.0, -0.6, 0.85]`.

**Regressor.** $\|\boldsymbol{x}\|_2$ only means something as the norm of an actual regressor, and
the integral reference needs the whole vector anyway, so we generate the AR input the simulation
uses and take the window whose norm is closest to the median.

**Regularization.** `sKF_L_integral_algorithm` regularizes its input as
`x_reg = sign(x)*(|x| + 1e-3)`, and it has to: the marginal mean divides by $x_{t,m}$, so an entry
near zero is a division by zero rather than a small perturbation. The closed forms therefore adopt
*its* regressor — we evaluate them at `x_t_reg` — so that both routes see the identical vector and
the ~0.12 % floor of `filters.py:322-330` does not contaminate the comparison.

In [ ]:
@njit
def numba_seed(seed):
    np.random.seed(seed)


M = 3
var_x = 1.0
var_v = 1e-3
b_eta = 5 * np.sqrt(var_v)
epsilon = 0.01
AR = np.array([1.0, -0.6, 0.85])
ho = np.sinc(np.linspace(0, 1.5, M)); ho = ho / np.linalg.norm(ho)
REG = 1e-3                       # filters.py:806, and filters.py:689

numba_seed(0)                    # std_behavior is @njit and keeps its own RNG state
np.random.seed(0)

N_sig = 2000
sig = std_behavior(N_sig, ho, var_x, var_v, AR, AR_settling_time(AR))
x_sig, d_sig = sig["x"], sig["d"]

# every regressor window, built exactly as the filters build theirs
windows = np.zeros((N_sig, M))
_w = np.zeros(M)
for k in range(N_sig):
    _w = shift(x_sig[k], _w)
    windows[k] = _w
windows = windows[M:]
norms = np.linalg.norm(windows, axis=1)

x_t = windows[int(np.argmin(np.abs(norms - np.median(norms))))].copy()
x_t_reg = np.sign(x_t) * (np.abs(x_t) + REG)
X = np.linalg.norm(x_t_reg)
w_prev = np.zeros(M)             # both updates are translation-invariant in w_{t-1}; checked below


def v_inf(X, b, eps, M):
    """Steady state of the sKF variance recursion (44)+(37), with v_eta = 2 b^2 (Table 1).

    v_t = v_{t-1} in (44) gives eps = (1/M) v~^2 X^2 / (v_eta + v~ X^2), a quadratic in v~.
    A nominal operating point, not a claim about the sKF-L recursion, whose own fixed point
    depends on e_t through b|e_t|. v~ is swept in Figure 5 anyway.
    """
    return 0.5 * M * (eps + np.sqrt(eps**2 + 8 * eps * b**2 / (M * X**2)))


v_tilde = v_inf(X, b_eta, epsilon, M)

print(f"x_t      = {x_t}")
print(f"x_t_reg  = {x_t_reg}")
print(f"X = ||x_t_reg||  = {X:.6f}")
print(f"  sqrt(M)*sigma_x = {np.sqrt(M * var_x):.6f}   (RMS reference, E||x||^2 = M var_x)")
print(f"  M*sigma_x       = {M * np.sqrt(var_x):.6f}   (not the 2-norm of anything here)")
print(f"  ||x_t|| quantiles 10/50/90 = {np.percentile(norms, [10, 50, 90]).round(4)}")
print(f"b_eta    = {b_eta:.6f}")
print(f"v_inf    = {v_tilde:.6f}   (v_tilde_0 in the notebook is 2.0, i.e. the transient)")

# empirical cross-check on v_inf
_p = np.void(("chk", epsilon, b_eta, 2.0), dtype=sKF_L_params)
_r = sKF_L_algorithm(N_sig, x_sig, d_sig, np.zeros(M), _p)
_v_emp = _r["v"][-1, 0]
print(f"  empirical converged v from sKF_L_algorithm: {_v_emp:.6f}  "
      f"(ratio {v_tilde / _v_emp:.2f})")
print("  The two differ by about 2x, and should: v_inf substitutes v_eta = 2 b_eta^2 for the")
print("  Laplacian scale, whereas the sKF-L recursion carries b_eta*|e_t|, and in steady state")
print("  |e_t| is of the order of the noise, well below b_eta. v_inf is a nominal operating")
print("  point; Figure 5 sweeps v~ across both values and two decades beyond.")

## 2. The gain functions

**Minorized**, from (50)-(51):

$$g_1^{\text{min}} = \frac{\tilde v\, e}{b|e| + \tilde v X^2}, \qquad
  g_2^{\text{min}} = -\frac{1}{M}\,\frac{\tilde v^2}{b|e| + \tilde v X^2}$$

**Exact**, from (63)-(64)/(69)-(70), with $\kappa_\varsigma = (\varsigma e - \tilde v X^2/b)/(\sqrt{\tilde v}X)$,
$\pi_\varsigma = \mathrm{softmax}(-\varsigma e/b + \log\Phi(\kappa_\varsigma))$, $h(\kappa)=\phi(\kappa)/\Phi(\kappa)$:

$$g_1^{\text{exact}} = \frac{\tilde v}{b}\Lambda_t - \frac{\sqrt{\tilde v}}{X}\Gamma_t, \qquad
  g_2^{\text{exact}} = \frac{D_t}{M}, \qquad
  D_t = \gamma^2(1-\Lambda_t^2) - 2\gamma\lambda(P_t - \Lambda_t\Gamma_t) - \lambda^2(Q_t + \Gamma_t^2)$$

Neither $g_1$ depends on $M$; only $g_2$ carries the $1/M$ of (19). The log-domain handling in
`_exact_scalars` mirrors `filters.py:369-380` and is load-bearing, not defensive: $\Phi(\kappa)$
underflows below $\kappa \approx -38$ and the two weights differ by $e^{2e/b}$, and both happen on
the same steps — the outliers this filter exists for.

In [ ]:
_SIGN = np.array([1.0, -1.0])     # the two mixture branches, varsigma = +1 and -1


def _exact_scalars(e, b, v, X):
    """kappa_+-, pi, and the four global scalars (62). Vectorized over e."""
    e = np.asarray(e, dtype=float)[..., None]
    kappa = (_SIGN * e - v * X**2 / b) / (np.sqrt(v) * X)
    log_Phi = log_ndtr(kappa)
    log_pi = -_SIGN * e / b + log_Phi
    pi = np.exp(log_pi - logsumexp(log_pi, axis=-1, keepdims=True))
    log_phi = -0.5 * kappa**2 - 0.5 * np.log(2 * np.pi)
    mills = np.exp(log_phi - log_Phi)
    return dict(
        kappa=kappa, pi=pi,
        Lambda=np.sum(_SIGN * pi, axis=-1),
        Gamma=np.sum(_SIGN * pi * mills, axis=-1),
        P=np.sum(pi * mills, axis=-1),
        Q=np.sum(pi * kappa * mills, axis=-1),
    )


def g1_min(e, b, v, X):
    e = np.asarray(e, dtype=float)
    return v * e / (b * np.abs(e) + v * X**2)


def g2_min(e, b, v, X, M):
    e = np.asarray(e, dtype=float)
    return -(v**2) / (M * (b * np.abs(e) + v * X**2))


def g1_exact(e, b, v, X):
    s = _exact_scalars(e, b, v, X)
    return v * s["Lambda"] / b - np.sqrt(v) * s["Gamma"] / X


def D_exact(e, b, v, X):
    s = _exact_scalars(e, b, v, X)
    g, l = v / b, np.sqrt(v) / X
    return (g**2 * (1 - s["Lambda"]**2)
            - 2 * g * l * (s["P"] - s["Lambda"] * s["Gamma"])
            - l**2 * (s["Q"] + s["Gamma"]**2))


def g2_exact(e, b, v, X, M):
    return D_exact(e, b, v, X) / M

## 3. The functions above are the filters

Without this check the figures would be about equations that may not be the ones `filters.py`
runs. We take ~40 real steps of each filter and rebuild every increment from `g1`/`g2`.

The tolerance is relative and set at `1e-10`, not `0`: the closed forms here use `X**2` where the
filters use `x @ x`, and `sqrt(x@x)**2 != x@x` in the last bits. Anything above `1e-10` would be a
genuine disagreement.

In [ ]:
def check_against_filters(N_check=40, tol=1e-10):
    wins = []
    _w = np.zeros(M)
    for k in range(N_check):
        _w = shift(x_sig[k], _w); wins.append(_w.copy())

    for algo, g1, g2, name in ((sKF_L_algorithm, g1_min, g2_min, "minorized (50)-(51)"),
                               (sKF_L_exact_algorithm, g1_exact, g2_exact, "exact (69)-(70)")):
        p = np.void((name[:20], epsilon, b_eta, 2.0), dtype=sKF_L_params)
        res = algo(N_check, x_sig[:N_check], d_sig[:N_check], np.zeros(M), p)
        w_hist, v_hist, e_hist = res["h"], res["v"][:, 0], res["e"]
        dw_err = v_err = scale_w = scale_v = 0.0
        for k in range(M, N_check - 1):
            xk = wins[k]; Xk = np.sqrt(xk @ xk); vt = v_hist[k] + epsilon
            dw = g1(e_hist[k], b_eta, vt, Xk) * xk
            vn = vt + g2(e_hist[k], b_eta, vt, Xk, M) * Xk**2
            dw_err = max(dw_err, np.max(np.abs(w_hist[k + 1] - w_hist[k] - dw)))
            v_err = max(v_err, abs(v_hist[k + 1] - vn))
            scale_w = max(scale_w, np.max(np.abs(dw))); scale_v = max(scale_v, abs(vn))
        rw, rv = dw_err / scale_w, v_err / scale_v
        print(f"  {name:22s}  rel err  dw {rw:.2e}   v {rv:.2e}")
        assert rw < tol and rv < tol, name


print("closed forms reproduce filters.py step by step:")
check_against_filters()

# parity: g1 odd in e, g2 even. Both follow from kappa_+(-e) = kappa_-(e).
_e = np.linspace(-60, 60, 2001) * b_eta
for nm, f in (("g1_min", g1_min), ("g1_exact", g1_exact)):
    a = f(_e, b_eta, v_tilde, X)
    print(f"  {nm:9s} odd   max|f(e)+f(-e)| = {np.max(np.abs(a + a[::-1])):.1e}")
for nm, f in (("g2_min", g2_min), ("g2_exact", g2_exact)):
    a = f(_e, b_eta, v_tilde, X, M)
    print(f"  {nm:9s} even  max|f(e)-f(-e)| = {np.max(np.abs(a - a[::-1])):.1e}")

## 4. The ground truth: one step of eq. (18), grid-aligned

`_integral_step` is the update body of `sKF_L_integral_algorithm` (`filters.py:817-874`) lifted out
of its time loop, with the two fixes of `grid-alignment-bias.md` applied:

1. `base_space` built symmetrically about a sample, so the centred composite noise $f^0_{\zeta_m}$
   really has mean 0 as Remark 3 requires — `np.arange(lo, hi, dx)` puts $t=0$ at a *fractional*
   index, and the $|x_n|$ scaling of (26) is then applied about the wrong point;
2. the convolution centre `len(f_eta)//2`, matching the output window of
   `integral_convolve_from_base_pdf`.

This matters here specifically: unfixed, the reference sits 1.4-3.4 % from *both* closed forms and
cannot tell them apart, which is the entire question. `filters.py` is left untouched — this is a
local copy.

The internal `x_reg` is kept exactly as it is, and the closed forms were moved to meet it
(section 1).

In [ ]:
def _integral_step(w0, x_raw, d_k, var_tilde, noise, dx_factor=1/20, min_std_deviations=6):
    """One update of the exact marginal (18) by numerical inversion. Returns (w_new, v_new).

    `noise` is ("gauss", var_eta) or ("lap", b_eta). Body copied from
    filters.py:817-874, with the two grid-alignment fixes marked below.
    """
    L = len(w0)
    x_reg = np.sign(x_raw) * (np.abs(x_raw) + REG)
    kind, scale = noise
    var_eta = scale if kind == "gauss" else 2 * scale**2   # filters.py:797

    w = w0.copy()
    var_theta = np.zeros(L)
    y_k = w @ x_reg
    worst_var_zeta = var_eta + (np.linalg.norm(x_reg)**2) * var_tilde
    mean_S = y_k - w * x_reg

    dx = np.sqrt(np.min([var_tilde, var_eta])) * dx_factor
    int_range = _get_integration_range(min_std_deviations, var_tilde, worst_var_zeta,
                                       d_k, x_reg, w, mean_S)

    # --- fix 1 (grid-alignment-bias.md): symmetric grid, t = 0 on a sample
    half = int(np.ceil(max(abs(int_range[0]), abs(int_range[1])) / dx))
    base_space = (np.arange(2 * half + 1) - half) * dx

    if kind == "gauss":
        f_eta = gaussian_pdf(base_space, np.void((0, var_eta), dtype=gaussian_params))
    else:
        f_eta = laplacian_pdf(base_space, np.void((0, scale), dtype=laplacian_params))
    base_pdf = gaussian_pdf(base_space, np.void((0, var_tilde), dtype=gaussian_params))

    for m in range(L):
        zeta_space = base_space + mean_S[m]
        likelihood_space = d_k - x_reg[m] * base_space
        freq_scalings = np.abs(x_reg[np.arange(0, L, 1) != m])
        f_zeta = integral_convolve_from_base_pdf(
            np.array([f_eta]), base_pdf, freq_scalings, np.zeros((L - 1,)),
            len(f_eta) // 2,                  # --- fix 2: centre matches the output window
            dx)
        likelihood = np.interp(likelihood_space, zeta_space, f_zeta, left=0, right=0)
        prior_m = gaussian_pdf(base_space, np.void((w[m], var_tilde), dtype=gaussian_params))
        with np.errstate(divide="ignore"):    # log(0) off the support, as in filters.py
            post_m = np.nan_to_num(np.exp(np.log(prior_m) + np.log(likelihood)))
        post_m /= np.trapezoid(post_m, dx=dx)
        w[m] = np.trapezoid(base_space * post_m, dx=dx)
        var_theta[m] = np.trapezoid(((base_space - w[m])**2) * post_m, dx=dx)

    return w, np.mean(var_theta)


def reference_gains(e_grid, x_raw, w0, var_tilde, noise, **kw):
    """g1 per coefficient and g2, read off eq. (18) as functions of e_t.

    e_t = y_t - x_t' w_{t-1}, so sweeping d_k at fixed x_t and w_{t-1} sweeps e_t.
    g1 divides by the *regularized* entry, which is what the closed forms are evaluated at.
    """
    x_reg = np.sign(x_raw) * (np.abs(x_raw) + REG)
    g1 = np.zeros((len(e_grid), len(w0)))
    g2 = np.zeros(len(e_grid))
    for i, e in enumerate(e_grid):
        w_new, v_new = _integral_step(w0, x_raw, w0 @ x_reg + e, var_tilde, noise, **kw)
        g1[i] = (w_new - w0) / x_reg
        g2[i] = (v_new - var_tilde) / (x_reg @ x_reg)
    return g1, g2

## 5. Validating the reference, before trusting it

Three checks. The first two are about the machinery, the third about the structure both equations
claim.

In [ ]:
# (a) Gaussian control. With a Gaussian likelihood the closed form (43) is exact, so this
#     validates the wrapper independently of the Laplacian question.
e_ctl = np.array([-3.0, -0.5, -0.05, 0.05, 0.5, 3.0]) * b_eta
var_eta_ctl = 2 * b_eta**2
g1_ctl, g2_ctl = reference_gains(e_ctl, x_t, w_prev, v_tilde, ("gauss", var_eta_ctl))
g1_ctl_closed = v_tilde * e_ctl / (var_eta_ctl + v_tilde * X**2)
g2_ctl_closed = -(v_tilde**2) / (M * (var_eta_ctl + v_tilde * X**2))
print("(a) Gaussian control against the closed-form sKF gain (43):")
print("     e/b      g1_num          g1_(43)         rel")
for i, e in enumerate(e_ctl):
    gm = g1_ctl[i].mean()
    print(f"   {e/b_eta:+6.2f}   {gm:+.8e}  {g1_ctl_closed[i]:+.8e}  {abs(gm-g1_ctl_closed[i])/abs(g1_ctl_closed[i]):.2e}")
print(f"     g2 rel err: {np.max(np.abs(g2_ctl - g2_ctl_closed)/abs(g2_ctl_closed)):.2e}")

# (b) w_{t-1} invariance: both updates are translation-invariant, so the reference must be too.
_g1_off, _g2_off = reference_gains(e_ctl, x_t, np.array([0.3, -0.7, 1.1]), v_tilde, ("lap", b_eta))
_g1_zero, _g2_zero = reference_gains(e_ctl, x_t, w_prev, v_tilde, ("lap", b_eta))
print(f"\n(b) w_(t-1) invariance: max rel change in g1 = "
      f"{np.max(np.abs(_g1_off - _g1_zero)/np.abs(_g1_zero)):.2e}, "
      f"g2 = {np.max(np.abs(_g2_off - _g2_zero)/np.abs(_g2_zero)):.2e}")

In [ ]:
# (c) Grid convergence, and the validity window it defines.
#     The reference has no exponential tilt (Appendix B), so it must fail at large |e_t| by the
#     containment condition (34). We find where instead of assuming a limit: run two grids and
#     call a point reliable where they agree to 1e-3.
RATIO = np.linspace(-16, 16, 161)          # e_t in units of b_eta
e_ref = RATIO * b_eta

g1_coarse, g2_coarse = reference_gains(e_ref, x_t, w_prev, v_tilde, ("lap", b_eta),
                                       dx_factor=1/20, min_std_deviations=6)
g1_fine, g2_fine = reference_gains(e_ref, x_t, w_prev, v_tilde, ("lap", b_eta),
                                   dx_factor=1/40, min_std_deviations=8)

g1_num = g1_fine.mean(axis=1)
g2_num = g2_fine
conv1 = np.abs(g1_coarse.mean(axis=1) - g1_num) / np.maximum(np.abs(g1_num), 1e-300)
conv2 = np.abs(g2_coarse - g2_num) / np.abs(g2_num)
spread = np.max(np.abs(g1_fine - g1_num[:, None]) / np.abs(g1_num[:, None]), axis=1)


def first_failure(ratio, conv, tol=1e-3):
    """Smallest |e_t|/b at which the two grids stop agreeing."""
    r = np.abs(ratio)
    for i in np.argsort(r):
        if r[i] > 1e-9 and conv[i] >= tol:
            return r[i]
    return np.inf


def lim_str(lim):
    return "the whole sweep" if not np.isfinite(lim) else f"|e_t|/b < {lim:g}"


LIM1, LIM2 = first_failure(RATIO, conv1), first_failure(RATIO, conv2)
OK1 = np.abs(RATIO) < LIM1
OK2 = np.abs(RATIO) < LIM2
NZ = np.abs(RATIO) > 1e-9          # g1_num -> 0 at e_t = 0; no relative error there
print(f"(c) reference converged over {lim_str(LIM1)} for g1, and {lim_str(LIM2)} for g2"
      f"   (swept |e_t|/b <= {RATIO.max():g})")
print(f"    max spread of g1_num across m, where converged: {np.max(spread[OK1 & NZ]):.2e}")
print(f"    (both equations predict dw_m proportional to x_(t,m); this is that claim, measured)")

## 6. Figures

Conventions, following the rest of the repo: solid = numerical reference, dashed = minorized,
dash-dot = exact, dotted = asymptote. Where the reference has not converged it is drawn faint and
the region is shaded — those points say nothing about the equations.

In [ ]:
PARAMS = (f"$M$ = {M}\n$b_\\eta$ = {b_eta:.4f}\n$\\tilde v_t$ = {v_tilde:.4f}\n"
          f"$\\|x_t\\|$ = {X:.4f}\n$\\varepsilon$ = {epsilon}")


def shade_unreliable(ax, lim):
    """Grey out where the numerical reference has not converged. No-op if it never failed."""
    if not np.isfinite(lim) or lim > RATIO.max():
        return
    for lo, hi in ((-RATIO.max() * 1.05, -lim), (lim, RATIO.max() * 1.05)):
        ax.axvspan(lo, hi, color="0.85", alpha=0.5, zorder=0, lw=0)


def param_box(ax, loc="lower right"):
    box = ax.legend(handles=[], title=PARAMS, loc=loc, fontsize=8, title_fontsize=8)
    ax.add_artist(box)


# --- Figure 1: the weight gain g1 -------------------------------------------------------------
fig, (axL, axR) = plt.subplots(1, 2, figsize=(13, 4.8), constrained_layout=True)

# drawn wide and pale so the two closed forms on top stay readable where they coincide
axL.plot(RATIO[OK1], g1_num[OK1], color="k", lw=4, alpha=0.22, solid_capstyle="round",
         label="numerical, eq. (18)")
axL.plot(RATIO, g1_min(e_ref, b_eta, v_tilde, X), color=COLORS[1], ls="--",
         label="minorized, eq. (50)")
axL.plot(RATIO, g1_exact(e_ref, b_eta, v_tilde, X), color=COLORS[0], ls="-.",
         label="exact, eq. (63)")
axL.axhline(v_tilde / b_eta, color="0.4", ls=":", lw=1)
axL.axhline(-v_tilde / b_eta, color="0.4", ls=":", lw=1, label=r"$\pm\tilde v/b_\eta$, eq. (73)")
axL.plot(RATIO, v_tilde * e_ref / (2 * b_eta**2 + v_tilde * X**2), color="0.6", lw=1,
         label="Gaussian sKF, eq. (43)")
shade_unreliable(axL, LIM1)
axL.set_xlabel(r"$e_t / b_\eta$"); axL.set_ylabel(r"$g_1$")
axL.set_title(r"$g_1$: the multiplier on $x_t$")
axL.set_xlim(RATIO.min(), RATIO.max())
axL.set_ylim(-1.6 * v_tilde / b_eta, 1.6 * v_tilde / b_eta)   # the Gaussian gain is unbounded
axL.grid(alpha=0.3); axL.legend(fontsize=8, loc="upper left")
param_box(axL)

e_log = np.logspace(-2, 3, 600) * b_eta
axR.semilogx(e_log / b_eta, g1_min(e_log, b_eta, v_tilde, X), color=COLORS[1], ls="--",
             label="minorized, eq. (50)")
axR.semilogx(e_log / b_eta, g1_exact(e_log, b_eta, v_tilde, X), color=COLORS[0], ls="-.",
             label="exact, eq. (63)")
axR.axhline(v_tilde / b_eta, color="0.4", ls=":", lw=1, label=r"$\tilde v/b_\eta$, eq. (73)")
if np.isfinite(LIM1):
    axR.axvline(LIM1, color="k", ls=":", lw=1, alpha=0.6)
    axR.annotate("reference\nvalid below", xy=(LIM1, 0.02), fontsize=7, ha="right",
                 xytext=(-4, 0), textcoords="offset points", color="0.3")
axR.set_xlabel(r"$|e_t| / b_\eta$"); axR.set_ylabel(r"$g_1$")
axR.set_title("Approach to the saturated limit (Remark 9)")
axR.grid(alpha=0.3, which="both"); axR.legend(fontsize=8, loc="upper left")

fig.suptitle("Figure 1 - weight gain against prediction error, at a fixed operating point")
plt.show()

In [ ]:
# --- Figure 2: the verdict --------------------------------------------------------------------
def rel_db(a, ref):
    with np.errstate(divide="ignore", invalid="ignore"):
        return 10 * np.log10(np.abs(a - ref) / np.abs(ref))


fig, (axL, axR) = plt.subplots(1, 2, figsize=(13, 4.8), constrained_layout=True, sharey=True)

axL.plot(RATIO[OK1 & NZ], rel_db(g1_exact(e_ref, b_eta, v_tilde, X), g1_num)[OK1 & NZ],
         color=COLORS[0], ls="-.", label="exact, eq. (63)")
axL.plot(RATIO[OK1 & NZ], rel_db(g1_min(e_ref, b_eta, v_tilde, X), g1_num)[OK1 & NZ],
         color=COLORS[1], ls="--", label="minorized, eq. (50)")
axL.plot(RATIO[OK1 & NZ], 10 * np.log10(conv1[OK1 & NZ]), color="0.6", lw=1,
         label="reference's own grid convergence")
shade_unreliable(axL, LIM1)
axL.set_title(r"$g_1$ against the numerical reference"); axL.set_ylabel("relative error (dB)")

axR.plot(RATIO[OK2], rel_db(g2_exact(e_ref, b_eta, v_tilde, X, M), g2_num)[OK2],
         color=COLORS[0], ls="-.", label="exact, eq. (64)")
axR.plot(RATIO[OK2], rel_db(g2_min(e_ref, b_eta, v_tilde, X, M), g2_num)[OK2],
         color=COLORS[1], ls="--", label="minorized, eq. (51)")
axR.plot(RATIO[OK2], 10 * np.log10(conv2[OK2]), color="0.6", lw=1,
         label="reference's own grid convergence")
shade_unreliable(axR, LIM2)
axR.set_title(r"$g_2$ against the numerical reference")

for ax, lim in ((axL, LIM1), (axR, LIM2)):
    ax.axhline(-60, color="k", ls=":", lw=1, alpha=0.5)
    ax.set_xlabel(r"$e_t / b_\eta$"); ax.grid(alpha=0.3)
    ax.legend(fontsize=8, loc="upper center", ncol=3, framealpha=0.9)
    ax.set_xlim(RATIO.min(), RATIO.max())
    ax.margins(y=0.22)
param_box(axL, loc="lower left")

fig.suptitle("Figure 2 - are the equations right? Relative error against the numerical inversion of eq. (18)")
plt.show()

In [ ]:
# --- Figure 3: the variance gain g2, and the variance it produces ------------------------------
fig, (axL, axR) = plt.subplots(1, 2, figsize=(13, 4.8), constrained_layout=True)

axL.plot(RATIO[OK2], g2_num[OK2], color="k", lw=1.6, label="numerical, eq. (18)")
axL.plot(RATIO, g2_min(e_ref, b_eta, v_tilde, X, M), color=COLORS[1], ls="--",
         label="minorized, eq. (51)")
axL.plot(RATIO, g2_exact(e_ref, b_eta, v_tilde, X, M), color=COLORS[0], ls="-.",
         label="exact, eq. (64)")
axL.axhline(0, color="0.4", lw=0.8)
shade_unreliable(axL, LIM2)
axL.set_xlabel(r"$e_t / b_\eta$"); axL.set_ylabel(r"$g_2$")
axL.set_title(r"$g_2$: the multiplier on $\|x_t\|^2$")
axL.set_xlim(RATIO.min(), RATIO.max()); axL.grid(alpha=0.3); axL.legend(fontsize=8)
param_box(axL, loc="upper right")

vt_min = v_tilde + g2_min(e_ref, b_eta, v_tilde, X, M) * X**2
vt_exa = v_tilde + g2_exact(e_ref, b_eta, v_tilde, X, M) * X**2
axR.plot(RATIO[OK2], (v_tilde + g2_num * X**2)[OK2], color="k", lw=1.6, label="numerical, eq. (18)")
axR.plot(RATIO, vt_min, color=COLORS[1], ls="--", label="minorized, eq. (51)")
axR.plot(RATIO, vt_exa, color=COLORS[0], ls="-.", label="exact, eq. (70)")
axR.axhline(v_tilde, color="0.4", ls=":", lw=1, label=r"$\tilde v_t$ (no update)")
shade_unreliable(axR, LIM2)
axR.set_xlabel(r"$e_t / b_\eta$"); axR.set_ylabel(r"$v_t$")
axR.set_title(r"$v_t = \tilde v_t + g_2\|x_t\|^2$")
axR.set_xlim(RATIO.min(), RATIO.max()); axR.grid(alpha=0.3); axR.legend(fontsize=8)

fig.suptitle("Figure 3 - variance gain, and the posterior variance it leaves")
plt.show()

# Section 5 states D_t <= 0 was observed in every numerical case but not established. Test it.
_e_wide = np.linspace(-400, 400, 8001) * b_eta
_worst = 0.0
_n = 0
for vv in (1e-3, 1e-2, v_tilde, 0.5, 2.0):
    for bb in (np.sqrt(var_v), b_eta, 1.0):
        for XX in (0.65, X, 2.4):
            # scale D_t against the size of the terms it is a difference of, so that
            # "positive" means a real violation and not cancellation roundoff
            scale = (vv / bb)**2 + (np.sqrt(vv) / XX)**2
            _worst = max(_worst, D_exact(_e_wide, bb, vv, XX).max() / scale)
            _n += 1
print(f"D_t over {_n} parameter combinations x {_e_wide.size} error values:")
print(f"  max D_t / (gamma^2 + lambda^2) = {_worst:+.2e}")
print(f"  -> D_t <= 0 to floating-point roundoff, so (70) always decreases the variance,")
print(f"     as Section 5 reports having observed but not established.")
print(f"min v_t over the plotted range: minorized {vt_min.min():.4e}, exact {vt_exa.min():.4e} (must be > 0)")

In [ ]:
# --- Figure 4: what the exact filter is doing ---------------------------------------------------
s = _exact_scalars(e_ref, b_eta, v_tilde, X)
fig, axes = plt.subplots(1, 3, figsize=(14, 4.2), constrained_layout=True)

axes[0].plot(RATIO, s["kappa"][:, 0], color=COLORS[0], label=r"$\kappa_+$")
axes[0].plot(RATIO, s["kappa"][:, 1], color=COLORS[1], label=r"$\kappa_-$")
axes[0].set_ylabel(r"$\kappa_\varsigma$"); axes[0].set_title("Mixture arguments, eq. (60)")

axes[1].plot(RATIO, s["pi"][:, 0], color=COLORS[0], label=r"$\pi_+$")
axes[1].plot(RATIO, s["Lambda"], color=COLORS[2], label=r"$\Lambda_t = \pi_+-\pi_-$")
axes[1].axhline(1, color="0.4", ls=":", lw=1); axes[1].axhline(-1, color="0.4", ls=":", lw=1)
axes[1].set_title(r"Weights and the saturating error $\Lambda_t \in [-1,1]$, eqs. (61)-(62)")

axes[2].plot(RATIO, s["Gamma"], color=COLORS[0], label=r"$\Gamma_t$")
axes[2].plot(RATIO, s["P"], color=COLORS[1], label=r"$P_t$")
axes[2].plot(RATIO, s["Q"], color=COLORS[3], label=r"$Q_t$")
axes[2].set_title("Inverse-Mills scalars, eq. (62)")

for ax in axes:
    ax.set_xlabel(r"$e_t / b_\eta$"); ax.grid(alpha=0.3); ax.legend(fontsize=8)
    ax.set_xlim(RATIO.min(), RATIO.max())

fig.suptitle(r"Figure 4 - the four global scalars that carry the Laplacian likelihood in eqs. (63)-(64)")
plt.show()

In [ ]:
# --- Figure 5: how the gains respond to the other three parameters --------------------------
# x-axis is e_t in absolute units, not e_t/b_eta: b_eta is one of the swept parameters, so a
# b-relative axis would mean something different in each curve of the middle column.
# Bottom row is v_t/v~_t, the fraction of the predicted variance the update keeps: g2 itself
# spans three decades across the v~ sweep and would be unreadable on one pair of axes.
e_fam = np.linspace(-40, 40, 1601) * b_eta
sweeps = [
    (r"$\tilde v_t$", [0.005, v_tilde, 0.2, 2.0], lambda val: dict(b=b_eta, v=val, X=X)),
    (r"$b_\eta$", [np.sqrt(var_v), b_eta, 5 * b_eta, 20 * b_eta],
     lambda val: dict(b=val, v=v_tilde, X=X)),
    (r"$\|x_t\|$", list(np.percentile(norms, [10, 50, 90])) + [2 * np.percentile(norms, 90)],
     lambda val: dict(b=b_eta, v=v_tilde, X=val)),
]

fig, axes = plt.subplots(2, 3, figsize=(15, 8), constrained_layout=True)
for col, (name, values, mk) in enumerate(sweeps):
    for i, val in enumerate(values):
        kw = mk(val); c = COLORS[i % len(COLORS)]
        lbl = f"{name} = {val:.4g}"
        axes[0, col].plot(e_fam, g1_min(e_fam, **kw), color=c, ls="--", lw=1)
        axes[0, col].plot(e_fam, g1_exact(e_fam, **kw), color=c, ls="-", label=lbl)
        keep = lambda g2f: 1 + g2f(e_fam, M=M, **kw) * kw["X"]**2 / kw["v"]
        axes[1, col].plot(e_fam, keep(g2_min), color=c, ls="--", lw=1)
        axes[1, col].plot(e_fam, keep(g2_exact), color=c, ls="-", label=lbl)
    axes[0, col].set_title(f"sweeping {name}")
    for row in (0, 1):
        axes[row, col].set_xlabel(r"$e_t$"); axes[row, col].grid(alpha=0.3)
        axes[row, col].legend(fontsize=7)
    axes[1, col].set_ylim(-0.02, 1.02)
axes[0, 0].set_ylabel(r"$g_1$")
axes[1, 0].set_ylabel(r"$v_t / \tilde v_t$   (variance kept)")
axes[0, 0].plot([], [], color="k", ls="-", label="exact")
axes[0, 0].plot([], [], color="k", ls="--", label="minorized")
axes[0, 0].legend(fontsize=7)
fig.suptitle("Figure 5 - solid = exact (63)-(64), dashed = minorized (50)-(51). Analytic only.")
fig.savefig(
    "figure5_gain_functions.png", dpi=300, bbox_inches="tight", facecolor="white"
)
plt.show()
plt.show()

In [ ]:
# --- Summary numbers, for the findings cell below ----------------------------------------------
sel1 = OK1 & NZ     # g1 vanishes at e_t = 0
sel2 = OK2          # g2 is even and nonzero at e_t = 0
rows = [("g1", "exact  (63)", g1_exact(e_ref, b_eta, v_tilde, X), g1_num, sel1),
        ("g1", "minor. (50)", g1_min(e_ref, b_eta, v_tilde, X), g1_num, sel1),
        ("g2", "exact  (64)", g2_exact(e_ref, b_eta, v_tilde, X, M), g2_num, sel2),
        ("g2", "minor. (51)", g2_min(e_ref, b_eta, v_tilde, X, M), g2_num, sel2)]
print("Relative error against the numerical inversion of eq. (18), over the converged window")
print(f"({lim_str(LIM1)} for g1, {lim_str(LIM2)} for g2; swept |e_t|/b <= {RATIO.max():g}):\n")
print(f"  {'':4s} {'equation':12s} {'median':>10s} {'max':>10s}")
for what, name, a, ref, s_ in rows:
    r = np.abs(a - ref)[s_] / np.abs(ref)[s_]
    print(f"  {what:4s} {name:12s} {np.median(r):10.2e} {r.max():10.2e}")

## 7. Findings

At the operating point of the `##### Laplacian Likelihood` cells — $M=3$, $b_\eta = 5\sqrt{\mathrm{var}_v} \approx 0.158$,
$\tilde v_t = v_\infty \approx 0.0438$, $\|x_t\| \approx 1.573$ — measured against the numerical
inversion of eq. (18) over the window where that reference is grid-converged:

| | equation | median rel. err | max rel. err |
|---|---|---|---|
| $g_1$ | **exact (63)** | 3.7e-07 | 1.5e-05 |
| $g_1$ | minorized (50) | 2.6e-01 | 3.4e-01 |
| $g_2$ | **exact (64)** | 4.1e-07 | 3.3e-03 |
| $g_2$ | minorized (51) | 2.7e-01 | 2.9e+02 |

**Equations (63)-(64) are confirmed.** The residual is at the $10^{-7}$ level and, in Figure 2,
the exact curve sits *on* the reference's own grid-convergence curve — that is, the disagreement
is the reference's numerical error, not an error in the equations. There is no $e_t$ in the tested
range where the exact form departs from the ground truth by more than the ground truth's own
accuracy. The structural claims hold too: $\Delta w_m \propto x_{t,m}$ to 1e-05 across $m$, $g_1$
is odd and $g_2$ even to machine precision, and $D_t \le 0$ to roundoff over 45 parameter
combinations, so (70) never increases the variance — which Section 5 reports observing but not
establishing.

**Equations (50)-(51) disagree with the ground truth by 26-34 %, and that is expected.** They are
not meant to be exact: §4.1 replaces the Laplacian log-density with a quadratic minorizer before
marginalizing, and §4.4 says so explicitly — the minorization "should be read as the choice of a
simpler update rather than as a necessary step". So the finding is not that (50)-(51) are wrong as
printed, but that the approximation they encode costs about a quarter of the gain across the whole
range of $e_t$, not only at the outliers. Figure 1 shows the shape of the cost: the minorized gain
tracks the exact one near $e_t = 0$ and then under-responds over the intermediate range before both
reach the same $\tilde v/b_\eta$ saturation, which is exactly the behaviour Remark 9 predicts
— $O(1/|e_t|)$ approach for the minorized form against a Gaussian tail for the exact one.

$g_2$ is where the minorization hurts most. Figure 3 shows the kink at $e_t = 0$ that $b_\eta|e_t|$
puts in the denominator of (51), against the smooth quadratic minimum of (64); and for
$|e_t|/b_\eta \gtrsim 8$ the two differ by more than two orders of magnitude, the minorized form
still removing variance where the exact one has all but stopped.

### Caveats

* The reference has no exponential tilt (Appendix B), so it fails at large $|e_t|$ by the
  containment condition (34). Measured here: converged over the whole $|e_t|/b_\eta \le 16$ sweep
  for $g_1$, and up to $|e_t|/b_\eta \approx 12$ for $g_2$. Beyond that the comparison is shaded
  out and says nothing. The $2.9\times10^2$ figure for $g_2$ sits near that edge.
* This is one operating point and one regressor. Figure 5 shows the two families stay
  qualitatively ordered the same way across two decades of $\tilde v_t$, $b_\eta$ and $\|x_t\|$,
  but the numbers in the table are for the point in the table.
* The reference used here is `_integral_step`, a local copy of `sKF_L_integral_algorithm` with the
  `grid-alignment-bias.md` fix applied. Without that fix the reference cannot separate the two
  closed forms at all, and none of the above would be measurable. `filters.py` is unchanged, so
  the existing notebook still carries the bias.